In [1]:
#Import the file
import pandas as pd

df = pd.read_excel('TradeStat-Eidb-Import-Country-wise.xlsx', dtype=str)
df.head()

,TradeStat->Eidb->Import->Country-wise,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Report Generated on: 11/10/2026 - Values in US...,NaN,NaN,NaN,NaN,NaN,NaN
1,S.No.,Country/Region,2024-2025,%Share,2025-2026,%Share,%Growth
2,1,AFGHANISTAN,689.81,0.0956,654.22,0.0843,-5.16
3,2,ALBANIA,6.9,0.001,4.78,0.0006,-30.63
4,3,ALGERIA,760.06,0.1054,358.65,0.0462,-52.81


In [2]:
#Remove the first row

df = df.iloc[1:]
df.head()

,TradeStat->Eidb->Import->Country-wise,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
1,S.No.,Country/Region,2024-2025,%Share,2025-2026,%Share,%Growth
2,1,AFGHANISTAN,689.81,0.0956,654.22,0.0843,-5.16
3,2,ALBANIA,6.9,0.001,4.78,0.0006,-30.63
4,3,ALGERIA,760.06,0.1054,358.65,0.0462,-52.81
5,4,AMERI SAMOA,0.01,0,0.01,0,-39.33


In [3]:
#Move first row to header
df.columns = df.iloc[0]
df = df[1:]
df.reset_index(drop=True, inplace=True)
df.head()

1,S.No.,Country/Region,2024-2025,%Share,2025-2026,%Share,%Growth
0,1,AFGHANISTAN,689.81,0.0956,654.22,0.0843,-5.16
1,2,ALBANIA,6.9,0.001,4.78,0.0006,-30.63
2,3,ALGERIA,760.06,0.1054,358.65,0.0462,-52.81
3,4,AMERI SAMOA,0.01,0,0.01,0,-39.33
4,5,ANDORRA,0.07,0,0.25,0,275.4


In [4]:
#Drop some unwanted columns
df = df.drop(columns=['2024-2025', '%Share', '%Growth'])
df.head()

1,S.No.,Country/Region,2025-2026
0,1,AFGHANISTAN,654.22
1,2,ALBANIA,4.78
2,3,ALGERIA,358.65
3,4,AMERI SAMOA,0.01
4,5,ANDORRA,0.25


In [5]:
#rename columns
df = df.rename(columns={'S.No.': 'country_key', 'Country/Region': 'country', '2025-2026': 'import_sales_usd_country_2025_2026'})
df.head()

1,country_key,country,import_sales_usd_country_2025_2026
0,1,AFGHANISTAN,654.22
1,2,ALBANIA,4.78
2,3,ALGERIA,358.65
3,4,AMERI SAMOA,0.01
4,5,ANDORRA,0.25


In [6]:
#read the regions file
df_region = pd.read_excel('eidb-regionwise-all-countries-import.xlsx', dtype=str)
df_region.head()

,Report Dated: 11 Oct 2026 Values in US $ Million,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,Country / Region,2024-2025,2025-2026,%Growth
1,Europe,"94,740.83","105,160.18",11.00
2,EU Countries,"60,681.15","66,186.33",9.07
3,AUSTRIA,949.09,"1,041.66",9.75
4,BELGIUM,"6,589.99","6,408.62",-2.75


In [7]:
#move first row to header
df_region.columns = df_region.iloc[0]
df_region = df_region[1:]

df_region.head()

,Country / Region,2024-2025,2025-2026,%Growth
1,Europe,"94,740.83","105,160.18",11.00
2,EU Countries,"60,681.15","66,186.33",9.07
3,AUSTRIA,949.09,"1,041.66",9.75
4,BELGIUM,"6,589.99","6,408.62",-2.75
5,BULGARIA,296.69,167.06,-43.69


In [8]:
#remove unwanted columns
df_region = df_region.drop(columns=['2024-2025', '2025-2026', '%Growth'])
df_region.head()

,Country / Region
1,Europe
2,EU Countries
3,AUSTRIA
4,BELGIUM
5,BULGARIA


In [9]:
#Remove rows that does not contain and country name
df_region = df_region[df_region['Country / Region'] != "India's Total Export"]
df_region = df_region[df_region['Country / Region'] != "%Share"]
df_region.head(15)

,Country / Region
1,Europe
2,EU Countries
3,AUSTRIA
4,BELGIUM
5,BULGARIA
6,CROATIA
7,CYPRUS
8,CZECH REPUBLIC
9,DENMARK
10,ESTONIA


In [10]:
df_region.to_excel('dim_region.xlsx', index=False)

In [14]:
#read updated table dim_region that I changed with excel now we have to merge it with df
df_new = pd.read_excel('dim_region.xlsx', dtype=str)
df_new.head()

,country,region,continent
0,AUSTRIA,EU Countries,Europe
1,BELGIUM,EU Countries,Europe
2,BULGARIA,EU Countries,Europe
3,CROATIA,EU Countries,Europe
4,CYPRUS,EU Countries,Europe


In [15]:
#remove leading and end white spaces
df_new['country'] = df_new['country'].str.strip()
df_new['region'] = df_new['region'].str.strip()
df_new['continent'] = df_new['continent'].str.strip()

In [16]:
df_new.head()

,country,region,continent
0,AUSTRIA,EU Countries,Europe
1,BELGIUM,EU Countries,Europe
2,BULGARIA,EU Countries,Europe
3,CROATIA,EU Countries,Europe
4,CYPRUS,EU Countries,Europe


In [17]:
#Merge df and df_new on country
merged_df = pd.merge(df,df_new, on ='country')
merged_df.head()

,country_key,country,import_sales_usd_country_2025_2026,region,continent
0,1,AFGHANISTAN,654.22,South Asia,Asia
1,2,ALBANIA,4.78,Other European Countries,Europe
2,3,ALGERIA,358.65,North Africa,Africa
3,4,AMERI SAMOA,0.01,Unspecified Region,NaN
4,5,ANDORRA,0.25,Unspecified Region,NaN


In [18]:
#Rearrange column
merged_df = merged_df[['country_key', 'country', 'region', 'continent', 'import_sales_usd_country_2025_2026']]

In [19]:
merged_df.to_excel('dim_country.xlsx', index=False)